# Clase 4 — Práctica: buenas prácticas de entrenamiento

Curso "Fundamentos de TensorFlow desde cero" — DIH DATAlife

En esta práctica trabajas con dos ejercicios:

1. Corregir el sobreajuste de un modelo añadiendo dropout y regularización L2
2. Guardar solo los pesos de un modelo entrenado y recargarlos en una arquitectura nueva

Completa las celdas marcadas con `# TODO`. Cuando termines cada ejercicio, despliega la solución para comprobar tu respuesta, pero intenta resolverlo por tu cuenta antes de mirarla.

## Preparación común

Ejecuta esta celda antes de empezar cualquiera de los dos ejercicios: carga el dataset IMDB (Internet Movie Database) de reseñas de cine, lo codifica en vectores multi-hot y define las funciones que vas a reutilizar.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
import matplotlib.pyplot as plt

print("Versión de TensorFlow:", tf.__version__)

NUM_WORDS = 10000
N_TRAIN = 10000
N_VAL = 5000

(train_data, train_labels), (test_data, test_labels) = tf.keras.datasets.imdb.load_data(num_words=NUM_WORDS)

def multi_hot_sequences(sequences, dimension):
    """Convierte cada reseña (lista de IDs de palabra) en un vector de
    'dimension' posiciones: 1.0 si la palabra aparece, 0.0 si no aparece."""
    multi_hot = np.zeros((len(sequences), dimension), dtype="float32")
    for i, word_indices in enumerate(sequences):
        multi_hot[i, word_indices] = 1.0
    return multi_hot

x_train = multi_hot_sequences(train_data[:N_TRAIN], NUM_WORDS)
y_train = np.asarray(train_labels[:N_TRAIN]).astype("float32")

x_val = multi_hot_sequences(test_data[:N_VAL], NUM_WORDS)
y_val = np.asarray(test_labels[:N_VAL]).astype("float32")

print("Forma de x_train:", x_train.shape)
print("Forma de x_val:", x_val.shape)


def build_baseline_model():
    """Modelo de referencia: dos capas densas de 16 neuronas."""
    return tf.keras.Sequential([
        layers.Dense(16, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])


def compile_and_fit(model, max_epochs=20, extra_callbacks=None):
    """Compila y entrena un modelo con early stopping incluido: si la
    pérdida de validación no mejora durante 5 épocas seguidas, detiene el
    entrenamiento y recupera los mejores pesos vistos."""
    model.compile(optimizer="adam",
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=["accuracy"])

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True)

    callbacks = [early_stop] + (extra_callbacks or [])

    history = model.fit(x_train, y_train,
                         epochs=max_epochs,
                         batch_size=512,
                         validation_data=(x_val, y_val),
                         callbacks=callbacks,
                         verbose=0)
    return history


def plot_histories(histories_dict, metric="loss"):
    plt.figure(figsize=(9, 6))
    for name, history in histories_dict.items():
        epochs = range(1, len(history.history[metric]) + 1)
        linea = plt.plot(epochs, history.history[metric],
                          label=f"{name} (entrenamiento)")
        plt.plot(epochs, history.history["val_" + metric], "--",
                  color=linea[0].get_color(), label=f"{name} (validación)")
    plt.xlabel("Época")
    plt.ylabel(metric)
    plt.legend()
    plt.show()

## Ejercicio 1 — Corrige el sobreajuste con dropout y L2

El modelo `build_overfitting_model` de abajo tiene dos capas densas de 512 neuronas: mucha más capacidad de la que este problema necesita. Entrénalo y observa cómo sobreajusta (la pérdida de validación se separa pronto de la de entrenamiento).

Tu tarea es completar `build_overfitting_model_regularizado` para que sea una versión regularizada del mismo modelo:

- Añade una capa `layers.Dropout(...)` después de cada capa densa oculta, con una tasa entre 0.2 y 0.5
- Aplica `kernel_regularizer=regularizers.l2(...)` a las dos capas densas ocultas, con un valor pequeño (por ejemplo, entre 0.0001 y 0.001)

**Pistas**

- Fíjate en cómo está construido `build_regularized_model` en el notebook de teoría de esta clase, pero no lo copies literalmente: escribe tú la arquitectura
- El dropout se coloca como una capa más dentro del `Sequential`, justo después de la capa densa a la que afecta
- No apliques dropout ni regularización a la última capa (la de salida con activación sigmoide)

In [ ]:
def build_overfitting_model():
    """Modelo con mucha más capacidad de la necesaria: va a sobreajustar."""
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu", input_shape=(NUM_WORDS,)),
        layers.Dense(512, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

history_overfitting = compile_and_fit(build_overfitting_model())
print(f"Modelo sin regularizar: entrenó {len(history_overfitting.epoch)} épocas")

In [ ]:
def build_overfitting_model_regularizado():
    """TODO: completa esta arquitectura para reducir el sobreajuste del
    modelo anterior usando dropout y regularización L2."""
    return tf.keras.Sequential([
        # TODO: capa densa de 512 neuronas con kernel_regularizer=regularizers.l2(...)
        # TODO: capa Dropout con una tasa entre 0.2 y 0.5
        # TODO: segunda capa densa de 512 neuronas con kernel_regularizer=regularizers.l2(...)
        # TODO: segunda capa Dropout
        layers.Dense(1, activation="sigmoid"),
    ])

history_regularizado = compile_and_fit(build_overfitting_model_regularizado())
print(f"Modelo regularizado: entrenó {len(history_regularizado.epoch)} épocas")

plot_histories({
    "sin_regularizar": history_overfitting,
    "regularizado": history_regularizado,
})

<details>
<summary>💡 Ver solución</summary>

```python
def build_overfitting_model_regularizado():
    """Versión regularizada: dropout + L2 en las dos capas ocultas."""
    return tf.keras.Sequential([
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001),
                     input_shape=(NUM_WORDS,)),
        layers.Dropout(0.5),
        layers.Dense(512, activation="relu",
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])

history_regularizado = compile_and_fit(build_overfitting_model_regularizado())
print(f"Modelo regularizado: entrenó {len(history_regularizado.epoch)} épocas")

plot_histories({
    "sin_regularizar": history_overfitting,
    "regularizado": history_regularizado,
})

# Con dropout (0.5) y L2 (0.001) el modelo regularizado tarda más en
# sobreajustar: la pérdida de validación se mantiene más cerca de la de
# entrenamiento durante más épocas que en el modelo original.
```
</details>

## Ejercicio 2 — Guarda solo los pesos y recárgalos en una arquitectura nueva

Entrena el modelo `modelo_original` que se define abajo y guarda **solo sus pesos** (no la arquitectura ni el optimizador) usando un callback `ModelCheckpoint`. Después, construye una **arquitectura nueva** (una instancia distinta, creada desde cero con `build_baseline_model()`) y carga en ella los pesos guardados.

Completa los `# TODO`:

1. Crea un callback `tf.keras.callbacks.ModelCheckpoint` que guarde solo los pesos en `checkpoint_path`
2. Evalúa la arquitectura nueva **antes** de cargar los pesos (debería rondar el 50% de precisión, como lanzar una moneda al aire)
3. Carga los pesos guardados en la arquitectura nueva con `load_weights`
4. Evalúa de nuevo y comprueba que la precisión ahora coincide con la del modelo original

**Pistas**

- `save_weights_only=True` es la opción clave del callback
- El nombre del archivo de pesos debe terminar en `.weights.h5`
- La arquitectura nueva tiene que ser exactamente igual a la del modelo original para que los pesos encajen; por eso reutilizamos `build_baseline_model()`

In [ ]:
checkpoint_path = "ejercicio2_pesos.weights.h5"

modelo_original = build_baseline_model()
modelo_original.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# TODO: crea un ModelCheckpoint que guarde SOLO los pesos en checkpoint_path
callback_checkpoint = None  # TODO

modelo_original.fit(x_train, y_train, epochs=8, batch_size=512,
                     validation_data=(x_val, y_val),
                     callbacks=[callback_checkpoint], verbose=0)

loss, acc = modelo_original.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo original ya entrenado: {acc:.2%}")

In [ ]:
# Arquitectura nueva, sin entrenar
modelo_nuevo = build_baseline_model()
modelo_nuevo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# TODO: evalúa modelo_nuevo ANTES de cargar los pesos y guarda el resultado
# en las variables loss_antes, acc_antes
loss_antes, acc_antes = None, None  # TODO
print(f"Precisión antes de cargar los pesos: {acc_antes:.2%}")

# TODO: carga en modelo_nuevo los pesos guardados en checkpoint_path

# TODO: evalúa modelo_nuevo DESPUÉS de cargar los pesos y guarda el resultado
# en las variables loss_despues, acc_despues
loss_despues, acc_despues = None, None  # TODO
print(f"Precisión después de cargar los pesos: {acc_despues:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
checkpoint_path = "ejercicio2_pesos.weights.h5"

modelo_original = build_baseline_model()
modelo_original.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# El callback guarda solo los pesos (save_weights_only=True) cada vez que
# mejora la pérdida de validación
callback_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_weights_only=True,
    save_best_only=True,
    monitor="val_loss",
    verbose=0,
)

modelo_original.fit(x_train, y_train, epochs=8, batch_size=512,
                     validation_data=(x_val, y_val),
                     callbacks=[callback_checkpoint], verbose=0)

loss, acc = modelo_original.evaluate(x_val, y_val, verbose=0)
print(f"Precisión del modelo original ya entrenado: {acc:.2%}")

# Arquitectura nueva, sin entrenar
modelo_nuevo = build_baseline_model()
modelo_nuevo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

loss_antes, acc_antes = modelo_nuevo.evaluate(x_val, y_val, verbose=0)
print(f"Precisión antes de cargar los pesos: {acc_antes:.2%}")

# Cargamos en la arquitectura nueva los pesos guardados del modelo original
modelo_nuevo.load_weights(checkpoint_path)

loss_despues, acc_despues = modelo_nuevo.evaluate(x_val, y_val, verbose=0)
print(f"Precisión después de cargar los pesos: {acc_despues:.2%}")

# acc_antes ronda el 50% (pesos aleatorios, equivalente a adivinar).
# acc_despues coincide con la precisión del modelo original: hemos
# reutilizado el conocimiento aprendido sin volver a entrenar.
```
</details>

## Para seguir practicando

Si quieres ir un poco más allá:

- Prueba distintos valores de dropout (0.2, 0.3, 0.5) en el ejercicio 1 y observa cómo cambia la curva de validación
- En el ejercicio 2, en lugar de `build_baseline_model()`, intenta cargar los pesos en una arquitectura con una capa densa intermedia añadida y comprueba qué error obtienes (los pesos no encajarán: es la forma más directa de entender por qué la arquitectura debe coincidir)
- Guarda el modelo completo del ejercicio 2 con `modelo_original.save("ejercicio2_modelo.keras")` y recupéralo con `tf.keras.models.load_model(...)`

Repasa el capítulo 06 del manual si necesitas refrescar la teoría antes de intentarlo.